# Documentação da base de ouro

**Etapa:** definição do contrato temporal

Este notebook registra fonte, unidade, período, frequência, horizonte, divisões temporais e versões do ambiente usadas em toda a análise.

**Entrada:** base modelável, arquivo bruto de ouro e protocolo  
**Resultado principal:** contrato único e reproduzível para os quatro modelos


## 1. Importações, caminhos e funções

In [1]:
from pathlib import Path
import hashlib, json, platform, sys, time, warnings
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from IPython.display import display

warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bases/grupo5/gold_daily_modeling.csv").exists())
BASE_PATH = ROOT / "bases/grupo5/gold_daily_modeling.csv"
OUT = ROOT / "analyses/grupo5/outputs"
OUT.mkdir(parents=True, exist_ok=True)
PROTOCOL_PATH = OUT / "protocol.json"
SEED = 42
np.random.seed(SEED)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_data():
    df = pd.read_csv(BASE_PATH, parse_dates=["DATE"]).sort_values("DATE").reset_index(drop=True)
    raw_path = ROOT / "bases/grupo5-tratamento/gold_daily_prices.csv"
    raw = pd.read_csv(raw_path, sep=None, engine="python", parse_dates=["DATE"])
    raw["EXPECTED_TARGET"] = pd.to_numeric(raw["VALUE"], errors="coerce").shift(-1)
    raw["TARGET_DATE"] = raw["DATE"].shift(-1)
    target_lookup = raw[["DATE", "TARGET_DATE", "EXPECTED_TARGET"]]
    df = df.merge(target_lookup, on="DATE", how="left", validate="one_to_one")
    assert df["DATE"].is_monotonic_increasing and not df["DATE"].duplicated().any()
    assert "TARGET_UP" not in df.columns and not df.isna().any().any()
    assert np.allclose(df["TARGET"], df["EXPECTED_TARGET"])
    return df.drop(columns="EXPECTED_TARGET")



In [2]:
def load_protocol(require_data_decisions=False):
    if not PROTOCOL_PATH.exists():
        raise RuntimeError("DECISÃO NECESSÁRIA: execute 01_documentacao_bases.ipynb primeiro.")
    protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    if protocol.get("protocol_version") != 2:
        raise RuntimeError("Protocolo desatualizado: execute novamente 01_documentacao_bases.ipynb.")
    if protocol["dataset_sha256"] != sha256(BASE_PATH):
        raise RuntimeError("A base congelada não corresponde ao hash do protocolo.")
    if require_data_decisions:
        missing = [k for k in ("source", "unit", "cleaning", "features", "seasonal_period") if not protocol["decisions"].get(k)]
        if missing:
            raise RuntimeError(f"DECISÃO NECESSÁRIA: complete as decisões {missing} antes de modelar.")
    return protocol

def save_protocol(protocol):
    PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8")

def origins(protocol, split):
    return protocol["origins"][split]

def target_dates(df, origin_ids):
    return df["TARGET_DATE"].iloc[origin_ids].to_numpy()


## 2. Contrato da base e protocolo temporal

In [3]:
SOURCE_PRIMARY = (
    "GitHub - dengyishuo/quantitative-finance/gold.daily.prices.csv; "
    "série subjacente Deutsche Bundesbank BBEX3.D.XAU.USD.EA.AC.C05"
)
SOURCE_URL = (
    "https://github.com/dengyishuo/quantitative-finance/blob/master/"
    "gold.daily.prices.csv"
)
TARGET_UNIT = "USD por onça fina de ouro (31,1034768 gramas)"

df = load_data()
n = len(df)
train_end = int(n * 0.70)
validation_end = int(n * 0.85)

candidate_features = [
    "GOLD_PRICE",
    "TREASURY_10Y",
    "FED_FUNDS_RATE",
    "DOW_SIN",
    "DOW_COS",
    "MONTH_SIN",
    "MONTH_COS",
    "GOLD_LAG_1",
    "GOLD_LAG_5",
    "GOLD_LAG_20",
    "GOLD_ROLLING_MEAN_5",
    "GOLD_ROLLING_STD_5",
]

sarimax_seasonal_orders = [
    [0, 0, 0, 5],
    [1, 0, 0, 5],
    [0, 0, 1, 5],
    [0, 1, 0, 5],
]



In [4]:
rf_directed_candidates = [
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": 1.0},
    {"n_estimators": 600, "max_depth": None, "min_samples_split": 5, "min_samples_leaf": 2, "max_features": 0.5},
    {"n_estimators": 900, "max_depth": None, "min_samples_split": 10, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 10, "min_samples_split": 5, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 600, "max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 2, "max_features": 1.0},
    {"n_estimators": 900, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 5, "max_features": 0.5},
    {"n_estimators": 300, "max_depth": 20, "min_samples_split": 10, "min_samples_leaf": 5, "max_features": 1.0},
    {"n_estimators": 600, "max_depth": 20, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 900, "max_depth": 20, "min_samples_split": 5, "min_samples_leaf": 2, "max_features": 0.5},
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 10, "min_samples_leaf": 2, "max_features": 0.5},
    {"n_estimators": 600, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"n_estimators": 900, "max_depth": 20, "min_samples_split": 5, "min_samples_leaf": 1, "max_features": 1.0},
]

model_search_spaces = {
    "SARIMAX": {
        "regular_orders": {"p": [0, 1, 2], "d": [0, 1], "q": [0, 1, 2]},
        "seasonal_orders": sarimax_seasonal_orders,
        "screening": "72 ajustes no treino por BIC; cinco menores BIC seguem para MAE walk-forward",
    },
    "Holt_Winters": {
        "trend": [None, "add", "mul"],
        "seasonal": [None, "add", "mul"],
        "damped_trend": [False, True],
        "seasonal_periods": [None, 5],
        "smoothing": "alpha, beta, gamma e phi otimizados internamente em cada janela",
    },
    "Random_Forest": {"strategy": "12 combinações dirigidas e predefinidas", "candidates": rf_directed_candidates},
    "PLS_Regression": {"n_components": "1 até min(12, n_features, n_train - 1)"},
}



In [5]:
selection_criterion = {
    "primary": "menor MAE de validação walk-forward",
    "tie_1": "MAE arredondado para 3 casas; escolher a configuração mais simples",
    "tie_2": "persistindo empate, menor tempo médio por origem",
    "simplicity": {
        "SARIMAX": "menor soma p+d+q+P+D+Q; depois menor ordem máxima",
        "Holt_Winters": "menos componentes ativos; sem amortecimento; aditivo antes de multiplicativo",
        "Random_Forest": "menor profundidade (None é ilimitada); depois menos árvores; depois folhas mais regulares",
        "PLS_Regression": "menor número de componentes",
    },
}

package_versions = {
    "python": platform.python_version(),
    **{name: version(name) for name in ["numpy", "pandas", "matplotlib", "scikit-learn", "statsmodels", "ipython", "nbformat"]},
}

previous_decisions = {
    "source": None,
    "unit": None,
    "cleaning": None,
    "features": None,
    "seasonal_period": None,
    "accepted_parameters": {},
}
previous_execution_phase = "tuning_validation_only"
previous_test_gate = {"status": "blocked", "unlock_requires": "parâmetros congelados e nova aprovação humana"}

if PROTOCOL_PATH.exists():
    saved_protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    if saved_protocol.get("dataset_sha256") == sha256(BASE_PATH):
        previous_decisions.update(saved_protocol.get("decisions", {}))
        previous_execution_phase = saved_protocol.get("execution_phase", previous_execution_phase)
        previous_test_gate = saved_protocol.get("test_gate", previous_test_gate)

previous_decisions["source"] = SOURCE_PRIMARY
previous_decisions["unit"] = TARGET_UNIT



In [6]:
protocol = {
    "protocol_version": 2,
    "dataset": "gold",
    "dataset_path": str(BASE_PATH.relative_to(ROOT)),
    "dataset_sha256": sha256(BASE_PATH),
    "seed": SEED,
    "horizon_observations": 1,
    "frequency": "próxima observação de mercado",
    "execution_phase": previous_execution_phase,
    "test_gate": previous_test_gate,
    "source_metadata": {
        "acquisition_url": SOURCE_URL,
        "underlying_series_id": "BBEX3.D.XAU.USD.EA.AC.C05",
        "description": "Gold price in London, afternoon fixing",
        "unit": TARGET_UNIT,
        "local_source_match": "12.009 datas e valores conferidos sem divergência",
    },
    "splits": {"train_end": train_end, "validation_end": validation_end, "n_rows": n},
    "origins": {"validation": list(range(train_end, validation_end, 5)), "test": list(range(validation_end, n, 5))},
    "candidate_features": candidate_features,
    "excluded_calendar_features": ["DAY_OF_WEEK", "MONTH"],
    "calendar_encoding": "seno e cosseno preservam a circularidade sem impor ordem artificial",
    "sarimax_exog": ["TREASURY_10Y", "FED_FUNDS_RATE", "DOW_SIN", "DOW_COS", "MONTH_SIN", "MONTH_COS"],
    "seasonal_candidates": [5, 20, 252],
    "seasonality_evidence": "STL no treino indicou força sazonal nula em 5, 20 e 252; m=5 é referência semanal, não presunção de sazonalidade",
    "model_search_spaces": model_search_spaces,
    "selection_criterion": selection_criterion,
    "package_versions": package_versions,
    "decisions": previous_decisions,
}

validation_origins = protocol["origins"]["validation"]
test_origins = protocol["origins"]["test"]
assert set(validation_origins).isdisjoint(test_origins)
assert max(validation_origins) < min(test_origins)
assert df.loc[max(validation_origins), "TARGET_DATE"] < df.loc[min(test_origins), "DATE"]

save_protocol(protocol)



In [7]:
origin_manifest = pd.concat(
    [
        pd.DataFrame({"split": split, "origin_index": ids})
        for split, ids in protocol["origins"].items()
    ],
    ignore_index=True,
)
origin_manifest["forecast_origin"] = df.loc[origin_manifest["origin_index"], "DATE"].dt.date.astype(str).to_numpy()
origin_manifest["target_date"] = df.loc[origin_manifest["origin_index"], "TARGET_DATE"].dt.date.astype(str).to_numpy()
origin_manifest["horizon"] = 1
origin_manifest.to_csv(OUT / "origin_manifest.csv", index=False)
pd.DataFrame(package_versions.items(), columns=["library", "version"]).to_csv(OUT / "environment_versions.csv", index=False)

summary = pd.DataFrame(
    {
        "campo": [
            "observações",
            "cobertura",
            "hash SHA-256",
            "externas",
            "origens de validação",
            "origens de teste",
        ],
        "valor": [
            n,
            f"{df.DATE.min().date()} a {df.DATE.max().date()}",
            protocol["dataset_sha256"],
            "TREASURY_10Y; FED_FUNDS_RATE",
            len(protocol["origins"]["validation"]),
            len(protocol["origins"]["test"]),
        ],
    }
)

display(summary)
display(pd.DataFrame({"feature candidata": candidate_features}))

if not SOURCE_PRIMARY or not TARGET_UNIT:
    raise RuntimeError("DECISÃO NECESSÁRIA: informe fonte primária e unidade; o protocolo foi salvo como rascunho.")

print("Contrato documental aprovado por você. Prossiga para limpeza/preparação.")

,campo,valor
0,observações,9864
1,cobertura,1968-04-29 a 2014-04-09
2,hash SHA-256,38622fe40a18c3e94b89971488468f762353ed4234234e...
3,externas,TREASURY_10Y; FED_FUNDS_RATE
4,origens de validação,296
5,origens de teste,296


,feature candidata
0,GOLD_PRICE
1,TREASURY_10Y
2,FED_FUNDS_RATE
3,DOW_SIN
4,DOW_COS
5,MONTH_SIN
6,MONTH_COS
7,GOLD_LAG_1
8,GOLD_LAG_5
9,GOLD_LAG_20


Contrato documental aprovado por você. Prossiga para limpeza/preparação.


## Interpretação

A base local foi conferida contra a referência: são 12.009 datas e valores. A previsão usa horizonte de uma observação de mercado, com 296 origens de validação e 296 de teste.

## Artefatos

Arquivos usados ou produzidos nesta etapa:

- `outputs/protocol.json`
- `outputs/environment_versions.csv`
- `outputs/origin_manifest.csv`